# Transformer input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30, 39, or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [1]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_transformer_imlive.py and updated transformer files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 39              # 30, 39, or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['axial']  # Add 'dq' to run the global-attention model too
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
EPOCHS = 200
BATCH_SIZE = 16
D_MODEL = 96
DEPTH = 4
NUM_HEADS = 6
DROPOUT = 0.1
WARMUP_EPOCHS = 5
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = False
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'train_transformer_imlive.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case39
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case39_dq


In [2]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_transformer_imlive.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE,
    '--d-model', str(D_MODEL), '--depth', str(DEPTH), '--num-heads', str(NUM_HEADS),
    '--dropout', str(DROPOUT), '--warmup-epochs', str(WARMUP_EPOCHS)]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/train_transformer_imlive.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case39 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case39_dq --models axial --variants ddc ddc_yb ddc_imag ddc_real --target dq --seeds 42 123 2026 --epochs 200 --batch-size 16 --loss-alpha 0.5 --device cuda --d-model 96 --depth 4 --num-heads 6 --dropout 0.1 --warmup-epochs 5
PyTorch 2.10.0+cu128 device cuda
Dataset(dq, X=(6837, 3, 39, 39), y=(6837, 1, 39, 39)) 
 Dataset(dq, X=(1465, 3, 39, 39), y=(1465, 1, 39, 39))

axial ddc seed=42: 902,881 parameters; AMP=True
epoch=  1 loss=0.17589 val_R2=0.8630679532780168 val_RMSE=1.888620 (23.1s)
epoch=  2 loss=0.15092 val_R2=0.8637730480262154 val_RMSE=1.883751 (22.6s)
epoch=  3 loss=0.14702 val_R2=0.8632290033449956 val_RMSE=1.887509 (22.7s)
epoch=  4 loss=0

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/train_transformer_imlive.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case39', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case39_dq', '--models', 'axial', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--seeds', '42', '123', '2026', '--epochs', '200', '--batch-size', '16', '--loss-alpha', '0.5', '--device', 'cuda', '--d-model', '96', '--depth', '4', '--num-heads', '6', '--dropout', '0.1', '--warmup-epochs', '5'], returncode=0)

In [3]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
axial ddc test: {'r2': {'mean': 0.8730899951234786, 'std_across_seeds': 0.00025356362534800504}, 'rmse': {'mean': 1.8182649164685347, 'std_across_seeds': 0.0018159224453245415}, 'mae': {'mean': 0.72857162763126, 'std_across_seeds': 0.003048115265013181}}
axial ddc_yb test: {'r2': {'mean': 0.8768135785806743, 'std_across_seeds': 0.0006038455746002617}, 'rmse': {'mean': 1.7913890788935836, 'std_across_seeds': 0.004392516379645565}, 'mae': {'mean': 0.7126196987645894, 'std_across_seeds': 0.00807093551136452}}
axial ddc_imag test: {'r2': {'mean': 0.8748699258118257, 'std_across_seeds': 8.647475003215014e-05}, 'rmse': {'mean': 1.8054697102135684, 'std_across_seeds': 0.0006239170777734249}, 'mae': {'mean': 0.724976146377592, 'std_across_seeds': 0.012029753488246673}}
axial ddc_real test: {'r2': {'mean': 0.8748580251338852, 'std_across_seeds': 9.616433339908611e-05}, 'rmse': {'mean': 1.8055555471060476, 'std_across_seeds': 0.0006937442418578963}, 'mae': {'me

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. Axial attention is the default. Global attention over B*B cells has much higher memory use; reduce batch size if selecting dq. See README.md for inference and recovery.
